In [125]:
import glob
import json
import yaml
import shutil
import numpy as np
import pandas as pd
import datetime
from ray import tune
import hashlib
import rampwf as rw
import ramphy as rh
from pathlib import Path
from kaggle.api.kaggle_api_extended import KaggleApi
#import altair as alt
#alt.renderers.enable('default')
pd.options.display.max_columns = 200
pd.options.display.max_rows = 1000
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
from bokeh.plotting import figure, show
from bokeh.io import output_notebook
from bokeh.models import DatetimeTickFormatter, Label
from bokeh.resources import CDN
from bokeh.embed import file_html
output_notebook()

kits_root = Path("/home/gpaolo/data/ramp-kits")

/tmp/ipykernel_2904872/2606921247.py:18: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


Loading BokehJS ...

In [126]:
def kaggle_select(kaggle_api, suffix, id):
    f_name = kaggle_api.string(getattr(id, "fileName"))
    select = f_name[5:5 + len(suffix)] == suffix
    select = select and kaggle_api.string(getattr(id, "publicScore")) != ''
    return select

def ramp_kit_plot(ramp_kit_dir):
    problem = rw.utils.assert_read_problem(ramp_kit_dir=ramp_kit_dir)
    score_names = [st.name for st in problem.score_types]
    score_type = problem.score_types[-1]
    valid_score_name = f'valid_{score_names[-1]}'
    metadata = json.load(open(Path(ramp_kit_dir) / "data" / "metadata.json"))
    kaggle_api = KaggleApi()
    kaggle_api.authenticate()
    
    action_f_names = glob.glob(f'{ramp_kit_dir}/actions/*')
    action_f_names.sort()
    ramp_program = []
    start_time = datetime.datetime(2024, 5, 8, 12, 30, 00)
    for action_f_name in action_f_names:
        f_name = Path(action_f_name).name
        if f_name > str(start_time):
            ramp_program.append(rh.actions.load_ramp_action(action_f_name))
    
    hyperopt_actions = [ra for ra in ramp_program if ra.start_time > start_time and ra.name == "hyperopt"]
    blend_actions = [ra for ra in ramp_program if ra.start_time > start_time and (ra.name == "blend" or ra.name == "bag_then_blend")]
    train_actions = [ra for ra in ramp_program if ra.start_time > start_time and ra.name == "train"]
    print(blend_actions[-1].__dict__)
    start_time = min([ra.start_time for ra in ramp_program if ra.name == "hyperopt"])
    stop_time = max([ra.stop_time for ra in ramp_program])
    print(f"{len(hyperopt_actions)} rounds done")
    
    color_dict = {
        "catboost": "#2ca02c",
        "lgbm": "#ff7f0e",
        "xgboost": "#9467bd",
        "knn": "#d62728",
        "transformer": "#1f77b4",
    }
    tooltips = [
        ("submission", "@submission"),
        ("score", "@valid_score"),
        ("run time [s]", "@runtime"),
    ]
    p = figure(
        title=f"AutoDS on {ramp_kit_dir}", x_axis_label='action time',
        x_axis_type="datetime", y_axis_label=f"valid {score_names[-1]}",
        tooltips=tooltips, width=1000, height=500,
    )
    ### Kaggle private leaderboard percentile rank lines
    quantile_start = 0.5
    quantile_resolution = 0.05
    try:
        public_leaderboard_scores = np.load(Path(ramp_kit_dir) / "data" / "public_leaderboard_scores.npy")
    except:
        pass
    try:
        private_leaderboard_scores = np.load(Path(ramp_kit_dir) / "data" / "private_leaderboard_scores.npy")
        if score_type.is_lower_the_better:
            private_leaderboard_ranks = np.clip(np.arange(0, quantile_start + 0.0001, quantile_resolution), 0, 1)
        else:
            private_leaderboard_ranks = np.clip(np.arange(quantile_start, 1.01, quantile_resolution), 0, 1)
        private_leaderboard_quantiles = np.quantile(private_leaderboard_scores, private_leaderboard_ranks)
        for qi, quantile in enumerate(private_leaderboard_quantiles):
            if score_type.is_lower_the_better:
                percentile_rank = int((1 - private_leaderboard_ranks[qi]) * 100)
            else:
                percentile_rank = int(private_leaderboard_ranks[qi] * 100)
            source = pd.DataFrame({
                "submission": [percentile_rank, percentile_rank],
                "time": [start_time, stop_time],
                "valid_score": [quantile, quantile],
            })
            p.line("time", "valid_score", line_width=1, color="#d62728", alpha=0.5, source=source)
            rank_text = Label(
                x=stop_time, y=quantile, text=str(percentile_rank), text_color="#d62728", text_alpha=0.5,
                text_align="left", text_font_size="6px", text_baseline="middle")
            p.add_layout(rank_text)     
    except:
        try:
            if score_type.is_lower_the_better:
                public_leaderboard_ranks = np.clip(np.arange(0, quantile_start + 0.0001, quantile_resolution), 0, 1)
            else:
                public_leaderboard_ranks = np.clip(np.arange(quantile_start, 1.01, quantile_resolution), 0, 1)
            public_leaderboard_quantiles = np.quantile(public_leaderboard_scores, public_leaderboard_ranks)
            for qi, quantile in enumerate(public_leaderboard_quantiles):
                if score_type.is_lower_the_better:
                    percentile_rank = int((1 - public_leaderboard_ranks[qi]) * 100)
                else:
                    percentile_rank = int(public_leaderboard_ranks[qi] * 100)
                source = pd.DataFrame({
                    "submission": [percentile_rank, percentile_rank],
                    "time": [start_time, stop_time],
                    "valid_score": [quantile, quantile],
                })
                p.line("time", "valid_score", line_width=1, color="#1f77b4", alpha=0.5, source=source)
                rank_text = Label(
                    x=stop_time, y=quantile, text=str(percentile_rank), text_color="#1f77b4", text_alpha=0.5,
                    text_align="left", text_font_size="6px", text_baseline="middle")
                p.add_layout(rank_text)     
        except:
            pass
    
    ### Hyperopt race action segment
    for ra in hyperopt_actions:
        if len(ra.mean_scores) > 0:
            submission = ra.kwargs["submission"]
            source = pd.DataFrame({
                "action time": [ra.start_time, ra.stop_time],
                "valid_score": [ra.mean_score, ra.mean_score],
                "submission": [submission, submission],
                "runtime": [int(ra.runtime.total_seconds()), int(ra.runtime.total_seconds())],
            })
            model_name = submission.split("_")[0]
            color = color_dict[model_name]
            p.line("action time", "valid_score", legend_label=model_name, line_width=3, color=color, source=source)
    
    ### Blended score after each action
    source = pd.DataFrame({
        "action time": [ra.start_time for ra in blend_actions],
        "valid_score": [ra.blended_score for ra in blend_actions],
        "submission": [ra.kwargs["submissions"] for ra in blend_actions],
        "runtime": [ra.runtime.total_seconds() for ra in blend_actions],
    })
    p.line("action time", "valid_score", legend_label='blend', line_width=3, color="black", source=source)
    
    ### Kaggle public and private scores
    try:            
        kaggle_submission_ids = kaggle_api.competition_submissions(competition=metadata["kaggle_name"])
        kaggle_submission_ids = [id for id in kaggle_submission_ids
                                 if not "_best_" in kaggle_api.string(getattr(id, "fileName"))]
        kaggle_public_scores = [kaggle_api.string(getattr(id, "publicScore")) for id in kaggle_submission_ids
                                if kaggle_select(kaggle_api, kit_suffix, id)]
        kaggle_private_scores = [kaggle_api.string(getattr(id, "privateScore")) for id in kaggle_submission_ids
                                 if kaggle_select(kaggle_api, kit_suffix, id)]
        kaggle_action_times = [kaggle_api.string(getattr(id, "description")) for id in kaggle_submission_ids
                               if kaggle_select(kaggle_api, kit_suffix, id)]
        kaggle_file_names = [kaggle_api.string(getattr(id, "fileName")) for id in kaggle_submission_ids
                             if kaggle_select(kaggle_api, kit_suffix, id)]
        source = pd.DataFrame({
            "action time": kaggle_action_times,
            "valid_score": kaggle_public_scores,
            "submission": kaggle_file_names,
        })
        source["action time"] = pd.to_datetime(source["action time"], format='mixed')
        p.line("action time", "valid_score", legend_label='public kaggle', line_width=3, color="#1f77b4", source=source)
        
        if "".join(kaggle_private_scores) != "":
            source = pd.DataFrame({
                "action time": kaggle_action_times,
                "valid_score": kaggle_private_scores,
                "submission": kaggle_file_names,
            })
            source["action time"] = pd.to_datetime(source["action time"], format='mixed')
            p.line("action time", "valid_score", legend_label='private kaggle', line_width=3, color="#d62728", source=source)
            
            final_time = max(source["action time"])
            final_score = float(source[source["action time"] == final_time]["valid_score"].to_numpy()[0])
            if score_type.is_lower_the_better:
                final_rank = np.less(float(final_score), private_leaderboard_scores).mean()
            else:
                final_rank = np.greater(float(final_score), private_leaderboard_scores).mean()
            final_rank_text = Label(
                x=final_time, y=final_score, text=str(round(100 * final_rank)), text_color="#d62728", text_alpha=1,
                text_align="left", text_font_size="18px", text_baseline="middle")
            p.add_layout(final_rank_text)
        else:
            final_time = max(source["action time"])
            final_score = float(source[source["action time"] == final_time]["valid_score"].to_numpy()[0])
            if score_type.is_lower_the_better:
                final_rank = np.less(float(final_score), public_leaderboard_scores).mean()
            else:
                final_rank = np.greater(float(final_score), public_leaderboard_scores).mean()
            final_rank_text = Label(
                x=final_time, y=final_score, text=str(round(100 * final_rank)), text_color="#1f77b4", text_alpha=1,
                text_align="left", text_font_size="18px", text_baseline="middle")
            p.add_layout(final_rank_text)     
    except Exception as e:
        print(e)
        pass
    
    ### Training actions after the end of the hyperopt race
    for ra in train_actions:
        submission = ra.kwargs["submission"]
        base_submission = submission[:-len("_hyperopt_0000000000")]
        if base_submission in color_dict.keys():
            source = pd.DataFrame({
                "action time": [ra.start_time, ra.stop_time],
                "valid_score": [ra.mean_score, ra.mean_score],
                "submission": [submission, submission],
                "runtime": [int(ra.runtime.total_seconds()), int(ra.runtime.total_seconds())],
            })
            p.line("action time", "valid_score", legend_label=base_submission, line_width=3, color=color_dict[base_submission], source=source)
    
    p.xaxis[0].formatter = DatetimeTickFormatter(hourmin = "%b%d %H:%M", minutes = "%H:%M", minsec = "%H:%M:%Ss")
    return p

In [127]:
ramp_kit = f"kaggle_cars_text_preprocessed"
versions = ["no_speed"]
number = "X"

for version in versions:
    kit_suffix = f"v{version}_n{number}"
    ramp_kit_dir = kits_root / f"{ramp_kit}_{kit_suffix}"

    p = ramp_kit_plot( ramp_kit_dir)
    show(p)

    plots_dir = Path(ramp_kit_dir) / "plots"
    plots_dir.mkdir(parents=False, exist_ok=True)
    html = file_html(p, CDN, "plot")
    with open(plots_dir / "summary_plot.html", "w") as f:
        f.write(html)

{'module': 'ramphy.actions', 'name': 'blend', 'args': (), 'kwargs': {'ramp_kit_dir': '/home/gpaolo/data/ramp-kits/kaggle_cars_text_preprocessed_vno_speed_nX', 'submissions': ['lgbm_hyperopt_85b3f2f763', 'lgbm_hyperopt_17d137a4ba', 'catboost_hyperopt_84df148d96', 'transformer_regressor_hyperopt_e1c91360a7', 'transformer_regressor_hyperopt_099aa2fa98', 'catboost_hyperopt_9ba0623716', 'transformer_regressor_hyperopt_1168c58a84', 'transformer_regressor_hyperopt_1cb58b322e', 'transformer_regressor_hyperopt_4bf867b08e', 'lgbm_hyperopt_433e78c651', 'catboost_hyperopt_fbb1654779', 'lgbm_hyperopt_aa862b0de8', 'catboost_hyperopt_01aec4c9bb'], 'fold_idxs': range(900, 903)}, 'start_time': datetime.datetime(2024, 9, 17, 10, 48, 49, 443496), 'stop_time': datetime.datetime(2024, 9, 17, 10, 48, 50, 827422), 'blended_score': 71561.75447844573, 'contributivities': {'lgbm_hyperopt_aa862b0de8': 269, 'catboost_hyperopt_84df148d96': 152, 'transformer_regressor_hyperopt_4bf867b08e': 122, 'lgbm_hyperopt_433e7

/home/gpaolo/miniforge3/envs/pangu/lib/python3.10/site-packages/urllib3/connectionpool.py:1063: InsecureRequestWarning: Unverified HTTPS request is being made to host 'proxyde.huawei.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


max() arg is an empty sequence


In [128]:
ramp_kit = f"kaggle_cars_text_preprocessed"
versions = ["combined"]
number = "X"

for version in versions:
    kit_suffix = f"v{version}_n{number}"
    ramp_kit_dir = kits_root / f"{ramp_kit}_{kit_suffix}"

    p = ramp_kit_plot( ramp_kit_dir)
    show(p)

    plots_dir = Path(ramp_kit_dir) / "plots"
    plots_dir.mkdir(parents=False, exist_ok=True)
    html = file_html(p, CDN, "plot")
    with open(plots_dir / "summary_plot.html", "w") as f:
        f.write(html)

{'module': 'ramphy.actions', 'name': 'bag_then_blend', 'args': (), 'kwargs': {'ramp_kit_dir': '/home/gpaolo/data/ramp-kits/kaggle_cars_text_preprocessed_vcombined_nX', 'submissions': ['lgbm_hyperopt_0aca58702a', 'lgbm_hyperopt_8832699d4e', 'lgbm_hyperopt_022a8cefdf', 'lgbm_hyperopt_6e81eed8ba', 'lgbm_hyperopt_cbf1442a10', 'lgbm_hyperopt_d0fe3b8cf5', 'lgbm_hyperopt_c34f31a7d9', 'lgbm_hyperopt_3eb03223bc', 'lgbm_hyperopt_5b60bc33e4', 'catboost_hyperopt_5ce8ad7758', 'lgbm_hyperopt_330c77e3a0', 'lgbm_hyperopt_ffdc52b4ea', 'lgbm_hyperopt_268f5d59ef', 'catboost_hyperopt_a46076e00b', 'lgbm_hyperopt_fe92d0cc57', 'lgbm_hyperopt_30f19b4950', 'lgbm_hyperopt_e27a9e15c8', 'lgbm_hyperopt_45bf47ce09', 'lgbm_hyperopt_7a4b1f2435', 'catboost_hyperopt_afdcccb599', 'lgbm_hyperopt_ae3f86af78', 'catboost_hyperopt_75c0eea4e9', 'lgbm_hyperopt_2c0eb1f9da', 'lgbm_hyperopt_13a89986fe', 'catboost_hyperopt_6232741e56'], 'fold_idxs': range(900, 931)}, 'start_time': datetime.datetime(2024, 9, 10, 10, 7, 14, 607070),

/home/gpaolo/miniforge3/envs/pangu/lib/python3.10/site-packages/urllib3/connectionpool.py:1063: InsecureRequestWarning: Unverified HTTPS request is being made to host 'proxyde.huawei.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


In [129]:
ramp_kit = f"kaggle_cars_text"
versions = ["tok_drop_knn"]
number = "X"

for version in versions:
    kit_suffix = f"v{version}_n{number}"
    ramp_kit_dir = kits_root / f"{ramp_kit}_{kit_suffix}"

    p = ramp_kit_plot( ramp_kit_dir)
    show(p)

    plots_dir = Path(ramp_kit_dir) / "plots"
    plots_dir.mkdir(parents=False, exist_ok=True)
    html = file_html(p, CDN, "plot")
    with open(plots_dir / "summary_plot.html", "w") as f:
        f.write(html)

{'module': 'ramphy.actions', 'name': 'bag_then_blend', 'args': (), 'kwargs': {'ramp_kit_dir': '/home/gpaolo/data/ramp-kits/kaggle_cars_text_vtok_drop_knn_nX', 'submissions': ['lgbm_hyperopt_2d75979963', 'lgbm_hyperopt_dd41964c53', 'lgbm_hyperopt_7e5f2b3fc1', 'knn_hyperopt_3928b3353c', 'lgbm_hyperopt_3177704368', 'lgbm_hyperopt_e677e2a5e2', 'lgbm_hyperopt_55f6f6996a', 'lgbm_hyperopt_857ce69540', 'lgbm_hyperopt_5b1e560062', 'lgbm_hyperopt_bf4a2b68af', 'lgbm_hyperopt_98a0c39e01', 'lgbm_hyperopt_0323b2f708', 'lgbm_hyperopt_58e8a8c4e1', 'lgbm_hyperopt_68929f5366', 'lgbm_hyperopt_1651cdc7d1', 'lgbm_hyperopt_d5b48e49db', 'lgbm_hyperopt_d011b70b05', 'lgbm_hyperopt_cf3ac61c66', 'lgbm_hyperopt_904937c9e9', 'lgbm_hyperopt_6e8775ea37', 'knn_hyperopt_0ac661555f', 'lgbm_hyperopt_ff13d0be07', 'lgbm_hyperopt_5bbac9ee58', 'catboost_hyperopt_f0ab6250e7', 'lgbm_hyperopt_74699d7520', 'lgbm_hyperopt_5a33aa116d', 'lgbm_hyperopt_a6c3167081', 'knn_hyperopt_dcebf0bf29'], 'fold_idxs': range(900, 931)}, 'start_t

/home/gpaolo/miniforge3/envs/pangu/lib/python3.10/site-packages/urllib3/connectionpool.py:1063: InsecureRequestWarning: Unverified HTTPS request is being made to host 'proxyde.huawei.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


In [130]:
ramp_kit = f"kaggle_cars_text"
versions = ["tok_drop_all"]
number = "X"

for version in versions:
    kit_suffix = f"v{version}_n{number}"
    ramp_kit_dir = kits_root / f"{ramp_kit}_{kit_suffix}"

    p = ramp_kit_plot( ramp_kit_dir)
    show(p)

    plots_dir = Path(ramp_kit_dir) / "plots"
    plots_dir.mkdir(parents=False, exist_ok=True)
    html = file_html(p, CDN, "plot")
    with open(plots_dir / "summary_plot.html", "w") as f:
        f.write(html)

{'module': 'ramphy.actions', 'name': 'bag_then_blend', 'args': (), 'kwargs': {'ramp_kit_dir': '/home/gpaolo/data/ramp-kits/kaggle_cars_text_vtok_drop_all_nX', 'submissions': ['lgbm_hyperopt_1d38c64ab9', 'catboost_hyperopt_894e3a68c6', 'lgbm_hyperopt_9f8dc948fa', 'catboost_hyperopt_cf739bbdaf', 'lgbm_hyperopt_fd1e1ac3a6', 'lgbm_hyperopt_7b6baa56fb', 'lgbm_hyperopt_9e596029ce', 'catboost_hyperopt_66cee68f64', 'lgbm_hyperopt_467d44ace6', 'catboost_hyperopt_c3d0f786bb', 'lgbm_hyperopt_7b396999da', 'lgbm_hyperopt_3ef5af3b19', 'catboost_hyperopt_78acb5cf5b', 'catboost_hyperopt_40fec8362d', 'catboost_hyperopt_998e789b94', 'lgbm_hyperopt_e5c3a679af', 'lgbm_hyperopt_5e896b9d5a', 'lgbm_hyperopt_fb9be61ea1', 'catboost_hyperopt_4868938b73', 'lgbm_hyperopt_1a7dba664f', 'catboost_hyperopt_6606e0d70d', 'catboost_hyperopt_3ecf784f35', 'lgbm_hyperopt_7b87f59aae', 'lgbm_hyperopt_d777e1da5d', 'lgbm_hyperopt_650ad56213', 'lgbm_hyperopt_4fd8cbde31', 'lgbm_hyperopt_9ed01e27e9', 'catboost_hyperopt_29c91b7b27

/home/gpaolo/miniforge3/envs/pangu/lib/python3.10/site-packages/urllib3/connectionpool.py:1063: InsecureRequestWarning: Unverified HTTPS request is being made to host 'proxyde.huawei.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
